In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import os
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import seaborn as sns
import folium
from folium.plugins import HeatMap
import glob
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from sklearn.metrics import pairwise_distances
from scipy.linalg import sqrtm
from scipy.special import digamma, loggamma
from math import pi

In [3]:
FilteredSynthetic = pd.read_csv('/Users/sankalpyadava/Desktop/Biocomplexity Institute/synthetic_filtered.csv')
NewestAdvan = pd.read_csv('/Users/sankalpyadava/Desktop/Biocomplexity Institute/NewestAdvan_VA.csv')

advan_agg = (NewestAdvan.groupby(['id_store', 'latitude', 'longitude', 'county'])['SUM_VISIT_COUNTS']
             .sum()
             .reset_index(name='visit_count_advan'))

synthetic_agg = (FilteredSynthetic.groupby(['lid', 'latitude', 'longitude', 'admin2'])
                  .size()
                  .reset_index(name='visit_count_synthetic'))

advan_agg['county'] = advan_agg['county'].astype(int)
synthetic_agg['admin2'] = synthetic_agg['admin2'].astype(int)

In [ ]:
dc_counties = [59, 13, 510, 107]

r = advan_agg[advan_agg['county'].isin(dc_counties)]
g = synthetic_agg[synthetic_agg['admin2'].isin(dc_counties)]

real_features = r[['latitude', 'longitude', 'visit_count_advan']].to_numpy(dtype=float)
fake_features = g[['latitude', 'longitude', 'visit_count_synthetic']].to_numpy(dtype=float)

nearest_k = 3

N, D = real_features.shape
M, _ = fake_features.shape

nbrs_R = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(real_features)
dist_R, _ = nbrs_R.kneighbors(real_features, nearest_k+1)

nbrs_G = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(fake_features)
dist_G, _ = nbrs_G.kneighbors(fake_features, nearest_k+1)

dist_RG_pairs = pairwise_distances(real_features, fake_features, n_jobs=-1)
dist_GR_pairs = pairwise_distances(fake_features, real_features, n_jobs=-1)

dist_RG, _ = nbrs_G.kneighbors(real_features, nearest_k+1)
dist_GR, _ = nbrs_R.kneighbors(fake_features, nearest_k+1)

def volume_of_unit_ball_log(d):
    return (d / 2) * np.log(pi) - loggamma((d / 2) + 1)

def cross_entropy(N, M, k, nu_k, d, eps=1e-10):
    nu_k = np.clip(nu_k, eps, None)
    return (1 / N) * np.sum(np.log(M) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(nu_k))

def entropy(N, k, rho_k, d, eps=1e-10):
    rho_k = np.clip(rho_k, eps, None)
    return (1 / N) * np.sum(np.log(N-1) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(rho_k))

def calc_precision(dist_R, dist_RG_pairs, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(np.any(G_in_radius, axis=0)) / M

def calc_density(dist_R, dist_RG_pairs, k, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(G_in_radius) / (k * M)

def calc_coverage(dist_R, dist_RG_pairs):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.mean(np.any(G_in_radius, axis=1))

def compute_fd(reps1, reps2, eps=1e-6):
    mu1, sigma1 = np.mean(reps1, axis=0), np.cov(reps1, rowvar=False)
    mu2, sigma2 = np.mean(reps2, axis=0), np.cov(reps2, rowvar=False)
    diff = mu1 - mu2
    try:
        covmean = sqrtm(sigma1.dot(sigma2))
        if np.iscomplexobj(covmean):
            covmean = covmean.real
    except ValueError:
        covmean = sqrtm(sigma1 + eps * np.eye(sigma1.shape[0])).dot(sqrtm(sigma2 + eps * np.eye(sigma2.shape[0])))
        covmean = covmean.real
    return np.dot(diff, diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean)

precision = calc_precision(dist_R, dist_RG_pairs, M)
recall = calc_precision(dist_G, dist_GR_pairs, N)
density = calc_density(dist_R, dist_RG_pairs, nearest_k, M)
coverage = calc_coverage(dist_R, dist_RG_pairs)

PCE = cross_entropy(M, N, nearest_k, dist_GR[:, nearest_k-1], D)
RCE = cross_entropy(N, M, nearest_k, dist_RG[:, nearest_k-1], D)
RE = entropy(N, nearest_k, dist_R[:, nearest_k], D)

fd = compute_fd(real_features, fake_features)

print(f"N = {N}, M = {M}, D = {D}")
print(f"\ndist_R:\n{dist_R}")
print(f"\ndist_G:\n{dist_G}")
print(f"\ndist_GR:\n{dist_GR}")
print(f"\ndist_RG:\n{dist_RG}")
print(f"\nprecision = {precision:.6f}")
print(f"recall    = {recall:.6f}")
print(f"density   = {density:.6f}")
print(f"coverage  = {coverage:.6f}")
print(f"\nPCE = {PCE:.6f}")
print(f"RCE = {RCE:.6f}")
print(f"RE  = {RE:.6f}")
print(f"\nfrechet_distance = {fd:.6f}")

In [6]:
cville_counties = [3]

r = advan_agg[advan_agg['county'].isin(cville_counties)]
g = synthetic_agg[synthetic_agg['admin2'].isin(cville_counties)]

real_features = r[['latitude', 'longitude', 'visit_count_advan']].to_numpy(dtype=float)
fake_features = g[['latitude', 'longitude', 'visit_count_synthetic']].to_numpy(dtype=float)

nearest_k = 10

N, D = real_features.shape
M, _ = fake_features.shape

nbrs_R = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(real_features)
dist_R, _ = nbrs_R.kneighbors(real_features, nearest_k+1)

nbrs_G = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(fake_features)
dist_G, _ = nbrs_G.kneighbors(fake_features, nearest_k+1)

dist_RG_pairs = pairwise_distances(real_features, fake_features, n_jobs=-1)
dist_GR_pairs = pairwise_distances(fake_features, real_features, n_jobs=-1)

dist_RG, _ = nbrs_G.kneighbors(real_features, nearest_k+1)
dist_GR, _ = nbrs_R.kneighbors(fake_features, nearest_k+1)

def volume_of_unit_ball_log(d):
    return (d / 2) * np.log(pi) - loggamma((d / 2) + 1)

def cross_entropy(N, M, k, nu_k, d):
    return (1 / N) * np.sum(np.log(M) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(nu_k))

def entropy(N, k, rho_k, d):
    return (1 / N) * np.sum(np.log(N-1) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(rho_k))

def calc_precision(dist_R, dist_RG_pairs, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(np.any(G_in_radius, axis=0)) / M

def calc_density(dist_R, dist_RG_pairs, k, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(G_in_radius) / (k * M)

def calc_coverage(dist_R, dist_RG_pairs):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.mean(np.any(G_in_radius, axis=1))

def compute_fd(reps1, reps2, eps=1e-6):
    mu1, sigma1 = np.mean(reps1, axis=0), np.cov(reps1, rowvar=False)
    mu2, sigma2 = np.mean(reps2, axis=0), np.cov(reps2, rowvar=False)
    diff = mu1 - mu2
    try:
        covmean = sqrtm(sigma1.dot(sigma2))
        if np.iscomplexobj(covmean):
            covmean = covmean.real
    except ValueError:
        covmean = sqrtm(sigma1 + eps * np.eye(sigma1.shape[0])).dot(sqrtm(sigma2 + eps * np.eye(sigma2.shape[0])))
        covmean = covmean.real
    return np.dot(diff, diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean)

precision = calc_precision(dist_R, dist_RG_pairs, M)
recall = calc_precision(dist_G, dist_GR_pairs, N)
density = calc_density(dist_R, dist_RG_pairs, nearest_k, M)
coverage = calc_coverage(dist_R, dist_RG_pairs)

PCE = cross_entropy(M, N, nearest_k, dist_GR[:, nearest_k-1], D)
RCE = cross_entropy(N, M, nearest_k, dist_RG[:, nearest_k-1], D)
RE = entropy(N, nearest_k, dist_R[:, nearest_k], D)

fd = compute_fd(real_features, fake_features)

print(f"N = {N}, M = {M}, D = {D}")
print(f"\ndist_R:\n{dist_R}")
print(f"\ndist_G:\n{dist_G}")
print(f"\ndist_GR:\n{dist_GR}")
print(f"\ndist_RG:\n{dist_RG}")
print(f"\nprecision = {precision:.6f}")
print(f"recall    = {recall:.6f}")
print(f"density   = {density:.6f}")
print(f"coverage  = {coverage:.6f}")
print(f"\nPCE = {PCE:.6f}")
print(f"RCE = {RCE:.6f}")
print(f"RE  = {RE:.6f}")
print(f"\nfrechet_distance = {fd:.6f}")

N = 1556, M = 2062, D = 3

dist_R:
[[0.00000000e+00 5.00000039e+00 1.80600000e+03 ... 6.48300000e+03
  7.07500000e+03 9.87600000e+03]
 [0.00000000e+00 4.80000006e+02 2.10800000e+03 ... 1.04810000e+04
  1.13810000e+04 1.37190000e+04]
 [0.00000000e+00 4.89000018e+02 2.33800000e+03 ... 7.37300000e+03
  8.23000000e+03 9.24900000e+03]
 ...
 [0.00000000e+00 1.02580781e+00 1.03775421e+00 ... 2.30011289e+01
  2.30019478e+01 2.50016003e+01]
 [0.00000000e+00 1.74000301e+02 2.64000100e+02 ... 7.47000034e+02
  8.70000022e+02 1.14300005e+03]
 [0.00000000e+00 2.16358027e-01 2.53403909e-01 ... 1.30029319e+01
  2.20018173e+01 2.30011733e+01]]

dist_G:
[[0.00000000e+00 1.52624072e-01 2.39605445e-01 ... 2.51170481e-01
  2.53929096e-01 2.65631623e-01]
 [0.00000000e+00 2.00118886e+00 7.00035034e+00 ... 3.70001525e+01
  3.90001689e+01 3.90002000e+01]
 [0.00000000e+00 1.36707866e-02 1.56380386e-02 ... 1.00193693e+00
  1.01219945e+00 1.03704045e+00]
 ...
 [0.00000000e+00 4.46209093e-03 5.61259250e-03 ... 3.8

In [ ]:
richmond_counties = [87, 41, 760, 86]

r = advan_agg[advan_agg['county'].isin(richmond_counties)]
g = synthetic_agg[synthetic_agg['admin2'].isin(richmond_counties)]

real_features = r[['latitude', 'longitude', 'visit_count_advan']].to_numpy(dtype=float)
fake_features = g[['latitude', 'longitude', 'visit_count_synthetic']].to_numpy(dtype=float)

#scaler = StandardScaler().fit(np.vstack([real_features, fake_features]))
#real_features = scaler.transform(real_features)
#fake_features = scaler.transform(fake_features)

nearest_k = 10

N, D = real_features.shape
M, _ = fake_features.shape

nbrs_R = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(real_features)
dist_R, _ = nbrs_R.kneighbors(real_features, nearest_k+1)

nbrs_G = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(fake_features)
dist_G, _ = nbrs_G.kneighbors(fake_features, nearest_k+1)

dist_RG_pairs = pairwise_distances(real_features, fake_features, n_jobs=-1)
dist_GR_pairs = pairwise_distances(fake_features, real_features, n_jobs=-1)

dist_RG, _ = nbrs_G.kneighbors(real_features, nearest_k+1)
dist_GR, _ = nbrs_R.kneighbors(fake_features, nearest_k+1)

def volume_of_unit_ball_log(d):
    return (d / 2) * np.log(pi) - loggamma((d / 2) + 1)

def cross_entropy(N, M, k, nu_k, d, eps=1e-10):
    nu_k = np.clip(nu_k, eps, None)
    return (1 / N) * np.sum(np.log(M) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(nu_k))

def entropy(N, k, rho_k, d, eps=1e-10):
    rho_k = np.clip(rho_k, eps, None)
    return (1 / N) * np.sum(np.log(N-1) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(rho_k))

def calc_precision(dist_R, dist_RG_pairs, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(np.any(G_in_radius, axis=0)) / M

def calc_density(dist_R, dist_RG_pairs, k, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(G_in_radius) / (k * M)

def calc_coverage(dist_R, dist_RG_pairs):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.mean(np.any(G_in_radius, axis=1))

def compute_fd(reps1, reps2, eps=1e-6):
    mu1, sigma1 = np.mean(reps1, axis=0), np.cov(reps1, rowvar=False)
    mu2, sigma2 = np.mean(reps2, axis=0), np.cov(reps2, rowvar=False)
    diff = mu1 - mu2
    try:
        covmean = sqrtm(sigma1.dot(sigma2))
        if np.iscomplexobj(covmean):
            covmean = covmean.real
    except ValueError:
        covmean = sqrtm(sigma1 + eps * np.eye(sigma1.shape[0])).dot(sqrtm(sigma2 + eps * np.eye(sigma2.shape[0])))
        covmean = covmean.real
    return np.dot(diff, diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean)

precision = calc_precision(dist_R, dist_RG_pairs, M)
recall = calc_precision(dist_G, dist_GR_pairs, N)
density = calc_density(dist_R, dist_RG_pairs, nearest_k, M)
coverage = calc_coverage(dist_R, dist_RG_pairs)

PCE = cross_entropy(M, N, nearest_k, dist_GR[:, nearest_k-1], D)
RCE = cross_entropy(N, M, nearest_k, dist_RG[:, nearest_k-1], D)
RE = entropy(N, nearest_k, dist_R[:, nearest_k], D)

fd = compute_fd(real_features, fake_features)

print(f"N = {N}, M = {M}, D = {D}")
print(f"\ndist_R:\n{dist_R}")
print(f"\ndist_G:\n{dist_G}")
print(f"\ndist_GR:\n{dist_GR}")
print(f"\ndist_RG:\n{dist_RG}")
print(f"\nprecision = {precision:.6f}")
print(f"recall    = {recall:.6f}")
print(f"density   = {density:.6f}")
print(f"coverage  = {coverage:.6f}")
print(f"\nPCE = {PCE:.6f}")
print(f"RCE = {RCE:.6f}")
print(f"RE  = {RE:.6f}")
print(f"\nfrechet_distance = {fd:.6f}")

In [ ]:
vabeach_counties = [810, 550, 700, 650, 710]

r = advan_agg[advan_agg['county'].isin(vabeach_counties)]
g = synthetic_agg[synthetic_agg['admin2'].isin(vabeach_counties)]

real_features = r[['latitude', 'longitude', 'visit_count_advan']].to_numpy(dtype=float)
fake_features = g[['latitude', 'longitude', 'visit_count_synthetic']].to_numpy(dtype=float)

#scaler = StandardScaler().fit(np.vstack([real_features, fake_features]))
#real_features = scaler.transform(real_features)
#fake_features = scaler.transform(fake_features)

nearest_k = 10

N, D = real_features.shape
M, _ = fake_features.shape

nbrs_R = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(real_features)
dist_R, _ = nbrs_R.kneighbors(real_features, nearest_k+1)

nbrs_G = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(fake_features)
dist_G, _ = nbrs_G.kneighbors(fake_features, nearest_k+1)

dist_RG_pairs = pairwise_distances(real_features, fake_features, n_jobs=-1)
dist_GR_pairs = pairwise_distances(fake_features, real_features, n_jobs=-1)

dist_RG, _ = nbrs_G.kneighbors(real_features, nearest_k+1)
dist_GR, _ = nbrs_R.kneighbors(fake_features, nearest_k+1)

def volume_of_unit_ball_log(d):
    return (d / 2) * np.log(pi) - loggamma((d / 2) + 1)

def cross_entropy(N, M, k, nu_k, d, eps=1e-10):
    nu_k = np.clip(nu_k, eps, None)
    return (1 / N) * np.sum(np.log(M) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(nu_k))

def entropy(N, k, rho_k, d, eps=1e-10):
    rho_k = np.clip(rho_k, eps, None)
    return (1 / N) * np.sum(np.log(N-1) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(rho_k))

def calc_precision(dist_R, dist_RG_pairs, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(np.any(G_in_radius, axis=0)) / M

def calc_density(dist_R, dist_RG_pairs, k, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(G_in_radius) / (k * M)

def calc_coverage(dist_R, dist_RG_pairs):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.mean(np.any(G_in_radius, axis=1))

def compute_fd(reps1, reps2, eps=1e-6):
    mu1, sigma1 = np.mean(reps1, axis=0), np.cov(reps1, rowvar=False)
    mu2, sigma2 = np.mean(reps2, axis=0), np.cov(reps2, rowvar=False)
    diff = mu1 - mu2
    try:
        covmean = sqrtm(sigma1.dot(sigma2))
        if np.iscomplexobj(covmean):
            covmean = covmean.real
    except ValueError:
        covmean = sqrtm(sigma1 + eps * np.eye(sigma1.shape[0])).dot(sqrtm(sigma2 + eps * np.eye(sigma2.shape[0])))
        covmean = covmean.real
    return np.dot(diff, diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean)

precision = calc_precision(dist_R, dist_RG_pairs, M)
recall = calc_precision(dist_G, dist_GR_pairs, N)
density = calc_density(dist_R, dist_RG_pairs, nearest_k, M)
coverage = calc_coverage(dist_R, dist_RG_pairs)

PCE = cross_entropy(M, N, nearest_k, dist_GR[:, nearest_k-1], D)
RCE = cross_entropy(N, M, nearest_k, dist_RG[:, nearest_k-1], D)
RE = entropy(N, nearest_k, dist_R[:, nearest_k], D)

fd = compute_fd(real_features, fake_features)

print(f"N = {N}, M = {M}, D = {D}")
print(f"\ndist_R:\n{dist_R}")
print(f"\ndist_G:\n{dist_G}")
print(f"\ndist_GR:\n{dist_GR}")
print(f"\ndist_RG:\n{dist_RG}")
print(f"\nprecision = {precision:.6f}")
print(f"recall    = {recall:.6f}")
print(f"density   = {density:.6f}")
print(f"coverage  = {coverage:.6f}")
print(f"\nPCE = {PCE:.6f}")
print(f"RCE = {RCE:.6f}")
print(f"RE  = {RE:.6f}")
print(f"\nfrechet_distance = {fd:.6f}")

In [4]:
roanoke = [161, 770]

r = advan_agg[advan_agg['county'].isin(roanoke)]
g = synthetic_agg[synthetic_agg['admin2'].isin(roanoke)]

real_features = r[['latitude', 'longitude', 'visit_count_advan']].to_numpy(dtype=float)
fake_features = g[['latitude', 'longitude', 'visit_count_synthetic']].to_numpy(dtype=float)

nearest_k = 10

N, D = real_features.shape
M, _ = fake_features.shape

nbrs_R = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(real_features)
dist_R, _ = nbrs_R.kneighbors(real_features, nearest_k+1)

nbrs_G = NearestNeighbors(n_neighbors=nearest_k+1, algorithm='auto', n_jobs=-1).fit(fake_features)
dist_G, _ = nbrs_G.kneighbors(fake_features, nearest_k+1)

dist_RG_pairs = pairwise_distances(real_features, fake_features, n_jobs=-1)
dist_GR_pairs = pairwise_distances(fake_features, real_features, n_jobs=-1)

dist_RG, _ = nbrs_G.kneighbors(real_features, nearest_k+1)
dist_GR, _ = nbrs_R.kneighbors(fake_features, nearest_k+1)

def volume_of_unit_ball_log(d):
    return (d / 2) * np.log(pi) - loggamma((d / 2) + 1)

def cross_entropy(N, M, k, nu_k, d):
    return (1 / N) * np.sum(np.log(M) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(nu_k))

def entropy(N, k, rho_k, d, eps=1e-10):
    rho_k = np.clip(rho_k, eps, None)
    return (1 / N) * np.sum(np.log(N-1) - digamma(k) + volume_of_unit_ball_log(d) + d * np.log(rho_k))

def calc_precision(dist_R, dist_RG_pairs, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(np.any(G_in_radius, axis=0)) / M

def calc_density(dist_R, dist_RG_pairs, k, M):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.sum(G_in_radius) / (k * M)

def calc_coverage(dist_R, dist_RG_pairs):
    radii_R = dist_R[:, -1]
    G_in_radius = (dist_RG_pairs <= radii_R[:, np.newaxis])
    return np.mean(np.any(G_in_radius, axis=1))

def compute_fd(reps1, reps2, eps=1e-6):
    mu1, sigma1 = np.mean(reps1, axis=0), np.cov(reps1, rowvar=False)
    mu2, sigma2 = np.mean(reps2, axis=0), np.cov(reps2, rowvar=False)
    diff = mu1 - mu2
    try:
        covmean = sqrtm(sigma1.dot(sigma2))
        if np.iscomplexobj(covmean):
            covmean = covmean.real
    except ValueError:
        covmean = sqrtm(sigma1 + eps * np.eye(sigma1.shape[0])).dot(sqrtm(sigma2 + eps * np.eye(sigma2.shape[0])))
        covmean = covmean.real
    return np.dot(diff, diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean)

precision = calc_precision(dist_R, dist_RG_pairs, M)
recall = calc_precision(dist_G, dist_GR_pairs, N)
density = calc_density(dist_R, dist_RG_pairs, nearest_k, M)
coverage = calc_coverage(dist_R, dist_RG_pairs)

PCE = cross_entropy(M, N, nearest_k, dist_GR[:, nearest_k-1], D)
RCE = cross_entropy(N, M, nearest_k, dist_RG[:, nearest_k-1], D)
RE = entropy(N, nearest_k, dist_R[:, nearest_k], D)

fd = compute_fd(real_features, fake_features)

print(f"N = {N}, M = {M}, D = {D}")
print(f"\ndist_R:\n{dist_R}")
print(f"\ndist_G:\n{dist_G}")
print(f"\ndist_GR:\n{dist_GR}")
print(f"\ndist_RG:\n{dist_RG}")
print(f"\nprecision = {precision:.6f}")
print(f"recall    = {recall:.6f}")
print(f"density   = {density:.6f}")
print(f"coverage  = {coverage:.6f}")
print(f"\nPCE = {PCE:.6f}")
print(f"RCE = {RCE:.6f}")
print(f"RE  = {RE:.6f}")
print(f"\nfrechet_distance = {fd:.6f}")

N = 3491, M = 3732, D = 3

dist_R:
[[0.00000000e+00 5.92600000e+03 7.80100000e+03 ... 1.84340000e+04
  2.10920000e+04 2.28800000e+04]
 [0.00000000e+00 1.82400000e+03 2.32200000e+03 ... 7.71300000e+03
  7.94800000e+03 1.01220000e+04]
 [0.00000000e+00 3.10000010e+02 6.66000002e+02 ... 2.30200000e+03
  2.88700000e+03 4.06800000e+03]
 ...
 [0.00000000e+00 1.71139560e-02 2.68590353e-02 ... 1.20000470e+01
  1.20002312e+01 2.40000467e+01]
 [0.00000000e+00 1.42357994e-01 1.54573557e-01 ... 1.10006194e+01
  1.10012404e+01 1.20008122e+01]
 [0.00000000e+00 2.41737996e-01 1.02268570e+00 ... 1.20021593e+01
  2.30017947e+01 2.40009401e+01]]

dist_G:
[[0.         0.00598845 0.01095947 ... 0.1767116  0.17816133 0.18141973]
 [0.         0.00117554 0.00353953 ... 0.01510963 0.02567012 0.04805333]
 [0.         0.40415543 0.4872338  ... 0.58553512 0.58553512 0.58568871]
 ...
 [0.         0.0039497  0.0053798  ... 0.02414197 0.02796091 0.03047744]
 [0.         0.         0.         ... 0.         0.       